<a href="https://colab.research.google.com/github/nahom-d54/AMHARIC-VOICE-CLONING/blob/main/tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi

Thu Sep 10 14:08:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4
VRAM: 14.56 GB


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/amharic_tts")

for directory in [
    "checkpoints",
    "datasets",
    "samples",
    "logs",
    "configs",
    "manifests",
    "audio",
    "tokens",
    "final_model",
]:
    (PROJECT / directory).mkdir(parents=True, exist_ok=True)

print(PROJECT)

/content/drive/MyDrive/amharic_tts


In [6]:
!python --version
!pip --version

Python 3.13.15
pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


In [7]:
!pip install -q -U omnivoice datasets soundfile librosa accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [7]:
!git clone https://github.com/k2-fsa/OmniVoice.git

Cloning into 'OmniVoice'...
remote: Enumerating objects: 558, done.
remote: Counting objects: 100% (241/241), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 558 (delta 165), reused 139 (delta 139), pack-reused 317 (from 3)
Receiving objects: 100% (558/558), 1.37 MiB | 19.54 MiB/s, done.
Resolving deltas: 100% (298/298), done.


In [8]:
%cd /content/OmniVoice

/content/OmniVoice


In [9]:
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.7 MB/s eta 0:00:00
  Building editable for omnivoice (pyproject.toml) ... done


In [10]:
import omnivoice

print("OmniVoice imported successfully")

OmniVoice imported successfully


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [11]:
!pip install -q soundfile

In [12]:
import torch
import soundfile as sf

from omnivoice import OmniVoice, OmniVoiceGenerationConfig

MODEL_ID = "african-low-resource/omnivoice-amharic"

model = OmniVoice.from_pretrained(
    MODEL_ID,
    device_map="cuda:0",
    dtype=torch.float16,
)

print("Model loaded")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Model loaded


In [14]:
text = "ሰላም፣ ይህ የአማርኛ የድምፅ ሙከራ ነው።"

audio = model.generate(
    text=text,
    language="Amharic",
    generation_config=OmniVoiceGenerationConfig(
        num_step=32,
        guidance_scale=2.0,
    ),
)

sf.write(
    "/content/drive/MyDrive/amharic_tts/samples/base_amharic.wav",
    audio[0],
    24000,
)

In [15]:
text = "ሰላም፣ ይህ የአማርኛ የድምፅ ሙከራ ነው።"

audio = model.generate(
    text=text,
    language="Amharic",
    generation_config=OmniVoiceGenerationConfig(
        num_step=32,
        guidance_scale=2.0,
    ),
)

sf.write(
    "/content/drive/MyDrive/amharic_tts/samples/base_amharic.wav",
    audio[0],
    24000,
)

In [16]:
from IPython.display import Audio, display

display(
    Audio(
        "/content/drive/MyDrive/amharic_tts/samples/base_amharic.wav"
    )
)

In [17]:
from google.colab import files

uploaded = files.upload()

Saving output.wav to output.wav


In [18]:
REFERENCE = next(iter(uploaded.keys()))

prompt = model.create_voice_clone_prompt(
    ref_audio=REFERENCE,
    ref_text="መጽሐፍ ማንበብ በጣም እወዳለሁ። በተለይ ስለ ታሪክና ስለ ቴክኖሎጂ የተጻፉ መጻሕፍትን።",
)

In [21]:
text = "መልካም አዲስ ዓመት! ይህ አዲስ ዓመት የሰላም፣ የጤና፣ የፍቅር እና የብልጽግና እንዲሆንልዎ ከልብ እመኛለሁ፤ ባለፉት ዓመታት ያጋጠሙዎትን ፈተናዎች ሁሉ በድል አጠናቀው፣ አዳዲስ የሕይወት ግቦችዎን የሚያሳኩበት የደስታ እና የስኬት ዘመን ይሁንልዎ። መልካም በዓል ከቤተሰብዎ ጋር ያሳልፉ!"

audio = model.generate(
    text=text,
    language="Amharic",
    voice_clone_prompt=prompt,
)

sf.write(
    "/content/drive/MyDrive/amharic_tts/samples/base_clone.wav",
    audio[0],
    24000,
)

In [22]:
display(
    Audio(
        "/content/drive/MyDrive/amharic_tts/samples/base_clone.wav"
    )
)

In [23]:
from datasets import load_dataset

dataset = load_dataset(
    "snapwre/amharic-speech"
)

dataset

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

data/train-00000-of-00028.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00028.parquet: downloading bytes:           |  0.00B            

...

data/test-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/13792 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1526 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1548 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes'],
        num_rows: 13792
    })
    validation: Dataset({
        features: ['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes'],
        num_rows: 1526
    })
    test: Dataset({
        features: ['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes'],
        num_rows: 1548
    })
})

In [25]:
for split in dataset:
    print("\nSPLIT:", split)
    print("Rows:", len(dataset[split]))
    print("Columns:")
    print(dataset[split].column_names)


SPLIT: train
Rows: 13792
Columns:
['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes']

SPLIT: validation
Rows: 1526
Columns:
['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes']

SPLIT: test
Rows: 1548
Columns:
['audio', 'clip_id', 'sentence', 'speaker_id', 'language', 'duration_s', 'speech_s', 'speech_start_s', 'speech_end_s', 'lufs', 'gender', 'age_band', 'region', 'sample_rate', 'up_votes', 'down_votes']


# Calculate Dataset Statistics

In [23]:
sample = dataset["train"][0]

for key, value in sample.items():
    if key == "audio":
        print(
            key,
            {
                "sampling_rate": value["sampling_rate"],
                "shape": value["array"].shape,
            }
        )
    else:
        print(key, value)

audio {'sampling_rate': 16000, 'shape': (173016,)}
clip_id clip_031090be2c68
sentence የምክር ቤቱ ጸሐፊ ሼህ ሁሴን በሽር በበኩላቸው፤ በዓሉን እስልምና በሚያዘው መሰረት በአብሮነትና በፍቅር ልናሳልፈው ይገባል ብለዋል፡፡
speaker_id spk_6674d883613f
language am
duration_s 10.812999725341797
speech_s 9.173999786376953
speech_start_s 0.7360000014305115
speech_end_s 10.812999725341797
lufs -28.399999618530273
gender male
age_band None
region None
sample_rate 16000
up_votes 3
down_votes 0


In [24]:
from collections import Counter

for split in dataset:
    counts = Counter(dataset[split]["speaker_id"])

    print(
        f"\n{split}: "
        f"{len(counts)} speakers"
    )

    print(
        "Min clips:",
        min(counts.values())
    )

    print(
        "Max clips:",
        max(counts.values())
    )

    print(
        "Average clips:",
        sum(counts.values()) / len(counts)
    )


train: 360 speakers
Min clips: 1
Max clips: 152
Average clips: 38.31111111111111

validation: 28 speakers
Min clips: 5
Max clips: 140
Average clips: 54.5

test: 105 speakers
Min clips: 1
Max clips: 138
Average clips: 14.742857142857142


# checking dupllicates

In [26]:
for split in dataset:
    texts = dataset[split]["sentence"]

    duplicates = len(texts) - len(set(texts))

    print(
        split,
        "duplicate transcripts:",
        duplicates
    )

train duplicate transcripts: 1339
validation duplicate transcripts: 17
test duplicate transcripts: 22


# Trying to normalize it

In [26]:
import unicodedata
import re

def normalize_amharic(text):
    if text is None:
        return ""

    text = str(text)

    # Unicode canonical normalization
    text = unicodedata.normalize("NFC", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

In [27]:
examples = [
    "ሰላም     እንዴት ነህ?",
    "  ይህ  የአማርኛ   ሙከራ ነው። "
]

for x in examples:
    print(normalize_amharic(x))

ሰላም እንዴት ነህ?
ይህ የአማርኛ ሙከራ ነው።


## Filter Bad Audio

- MIN_DURATION = 1.5
- MAX_DURATION = 30.0
- MIN_SPEECH = 0.8

In [28]:
MIN_DURATION = 1.5
MAX_DURATION = 30.0
MIN_SPEECH = 0.8
def valid_sample(row):
    text = normalize_amharic(row["sentence"])
    if not text:
        return False

    if row["duration_s"] < MIN_DURATION:
        return False

    if row["duration_s"] > MAX_DURATION:
        return False

    if row["speech_s"] < MIN_SPEECH:
        return False

    return True

In [29]:
for split in dataset:
    valid = sum(
        valid_sample(dataset[split][i])
        for i in range(len(dataset[split]))
    )

    print(
        split,
        valid,
        "/",
        len(dataset[split])
    )

KeyboardInterrupt: 

## OmniVoice's training pipeline expects JSONL entries of the form:
```
{
  "id": "...",
  "audio_path": "...",
  "text": "...",
  "language_id": "am"
}
```

In [30]:
import json
import os
import soundfile as sf
from pathlib import Path

AUDIO_DIR = PROJECT / "audio"

for split in ["train", "validation", "test"]:
    (AUDIO_DIR / split).mkdir(
        parents=True,
        exist_ok=True
    )

In [31]:
def export_split(split_name):
    ds = dataset[split_name]

    manifest_path = (
        PROJECT
        / "manifests"
        / f"{split_name}.jsonl"
    )

    with open(
        manifest_path,
        "w",
        encoding="utf-8"
    ) as manifest:

        for i in range(len(ds)):

            row = ds[i]

            if not valid_sample(row):
                continue

            text = normalize_amharic(
                row["sentence"]
            )

            clip_id = row["clip_id"]

            audio_path = (
                AUDIO_DIR
                / split_name
                / f"{clip_id}.wav"
            )

            audio = row["audio"]

            sf.write(
                audio_path,
                audio["array"],
                audio["sampling_rate"]
            )

            item = {
                "id": clip_id,
                "audio_path": str(
                    audio_path.resolve()
                ),
                "text": text,
                "language_id": "am",
            }

            manifest.write(
                json.dumps(
                    item,
                    ensure_ascii=False
                ) + "\n"
            )

    return manifest_path

In [34]:
train_manifest = export_split("train")
val_manifest = export_split("validation")
test_manifest = export_split("test")

print(train_manifest)
print(val_manifest)
print(test_manifest)

KeyboardInterrupt: 

save:

- checkpoints
- tokens
- models

back to Drive.

In [35]:
!mkdir -p /content/amharic_tts

In [36]:
!cp -r /content/drive/MyDrive/amharic_tts/audio /content/amharic_tts/

In [37]:
!cp -r /content/drive/MyDrive/amharic_tts/manifests /content/amharic_tts/

# Proof-of-Concept Dataset

- check if omni voice can improve the amharic quality with this corpus

In [38]:
from collections import defaultdict

speaker_to_rows = defaultdict(list)

for i in range(len(dataset["train"])):
    row = dataset["train"][i]

    speaker_to_rows[
        row["speaker_id"]
    ].append(i)

In [39]:
selected_speakers = list(
    speaker_to_rows.keys()
)[:10]

print(selected_speakers)

['spk_6674d883613f', 'spk_2781999af248', 'spk_580fd8276ca4', 'spk_efb391a67d78', 'spk_9c054ad076f4', 'spk_f2ab2b0e7cc9', 'spk_785eaafc249b', 'spk_543fa508df52', 'spk_a9e5bcbd13c3', 'spk_6917a9efa8b0']


In [40]:
%cd /content/OmniVoice

/content/OmniVoice


In [41]:
!mkdir -p /content/amharic_tts/tokens/train
!mkdir -p /content/amharic_tts/tokens/dev

In [44]:
!ls /content/amharic_tts/manifests

test.jsonl  train.jsonl  validation.jsonl


In [13]:
!CUDA_VISIBLE_DEVICES=0 python -m omnivoice.scripts.extract_audio_tokens \
    --input_jsonl /content/amharic_tts/manifests/train.jsonl \
    --tar_output_pattern "/content/amharic_tts/tokens/train/shard-%06d.tar" \
    --jsonl_output_pattern "/content/amharic_tts/tokens/train/shard-%06d.jsonl" \
    --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
    --nj_per_gpu 1 \
    --loader_workers 1 \
    --shuffle True

2026-09-10 17:10:16,365 INFO [extract_audio_tokens.py:341] Input mode: raw JSONL (/content/amharic_tts/manifests/train.jsonl)
2026-09-10 17:10:16,368 INFO [extract_audio_tokens.py:399] Adjusted samples_per_shard from 1000 to 240 to meet min_num_shards=32 (total_samples=7711)
2026-09-10 17:10:16,391 INFO [extract_audio_tokens.py:427] GPU count: 1, Processes per GPU: 1, Total processes: 1
Extracting Audio Tokens:   0% 0/7711 [00:00<?, ?it/s]2026-09-10 17:10:24,564 INFO [extract_audio_tokens.py:548] Submitting tasks... (1 workers)
2026-09-10 17:10:39,952 INFO [_client.py:1025] [Worker 54589] HTTP Request: HEAD https://huggingface.co/eustlb/higgs-audio-v2-tokenizer/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-10 17:10:39,952 WARNING [_http.py:955] [Worker 54589] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-10 17:10:40,045 INFO [_client.py:1025] [Worker 54589] HTTP Req

In [17]:
!CUDA_VISIBLE_DEVICES=0 python -m omnivoice.scripts.extract_audio_tokens \
    --input_jsonl /content/amharic_tts/manifests/validation.jsonl \
    --tar_output_pattern "/content/amharic_tts/tokens/dev/shard-%06d.tar" \
    --jsonl_output_pattern "/content/amharic_tts/tokens/dev/shard-%06d.jsonl" \
    --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
    --nj_per_gpu 1 \
    --loader_workers 2 \
    --shuffle True

2026-09-10 18:05:12,981 INFO [extract_audio_tokens.py:341] Input mode: raw JSONL (/content/amharic_tts/manifests/validation.jsonl)
2026-09-10 18:05:12,983 INFO [extract_audio_tokens.py:399] Adjusted samples_per_shard from 1000 to 47 to meet min_num_shards=32 (total_samples=1526)
2026-09-10 18:05:13,010 INFO [extract_audio_tokens.py:427] GPU count: 1, Processes per GPU: 1, Total processes: 1
Extracting Audio Tokens:   0% 0/1526 [00:00<?, ?it/s]2026-09-10 18:05:20,712 INFO [extract_audio_tokens.py:548] Submitting tasks... (1 workers)
2026-09-10 18:05:41,221 INFO [_client.py:1025] [Worker 68575] HTTP Request: HEAD https://huggingface.co/eustlb/higgs-audio-v2-tokenizer/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-10 18:05:41,222 WARNING [_http.py:955] [Worker 68575] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-10 18:05:41,303 INFO [_client.py:1025] [Worker 68575] HTTP

In [18]:
import json

data_config = {
    "train": [
        {
            "language_id": "am",
            "manifest_path": [
                "/content/amharic_tts/tokens/train/data.lst"
            ],
            "repeat": 1
        }
    ],
    "dev": [
        {
            "language_id": "am",
            "manifest_path": [
                "/content/amharic_tts/tokens/dev/data.lst"
            ],
            "repeat": 1
        }
    ]
}

with open(
    "/content/amharic_tts/data_config.json",
    "w"
) as f:
    json.dump(
        data_config,
        f,
        indent=2
    )

In [52]:
import json

train_config = {
    "llm_name_or_path": "Qwen/Qwen3-0.6B",
    "audio_vocab_size": 1025,
    "audio_mask_id": 1024,
    "num_audio_codebook": 8,
    "audio_codebook_weights": [8, 8, 6, 6, 4, 4, 2, 2],

    "drop_cond_ratio": 0.1,
    "prompt_ratio_range": [0.0, 0.3],
    "mask_ratio_range": [0.0, 1.0],
    "language_ratio": 0.8,

    "use_pinyin_ratio": 0.0,
    "instruct_ratio": 0.0,
    "only_instruct_ratio": 0.0,

    "resume_from_checkpoint": None,
    "init_from_checkpoint": "african-low-resource/omnivoice-amharic",

    "learning_rate": 1e-5,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,

    "steps": 1000,
    "seed": 42,

    "warmup_type": "ratio",
    "warmup_ratio": 0.05,
    "warmup_steps": 0,

    # MEMORY SETTINGS
    "batch_tokens": 1024,
    "gradient_accumulation_steps": 4,
    "max_sample_tokens": 1200,
    "max_batch_size": 8,
    "num_workers": 1,

    "mixed_precision": "fp16",
    "allow_tf32": True,

    "logging_steps": 25,
    "eval_steps": 250,
    "save_steps": 250,
    "keep_last_n_checkpoints": 3,

    "attn_implementation": "sdpa"
}

config_path = "/content/amharic_tts/train_config_poc.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(train_config, f, indent=2)

print(f"Saved to: {config_path}")

Saved to: /content/amharic_tts/train_config_poc.json


In [20]:
%cd /content/OmniVoice

/content/OmniVoice


In [22]:
!find /content/amharic_tts -maxdepth 2 -type f -print

/content/amharic_tts/data_config.json
/content/amharic_tts/tokens/data.lst
/content/amharic_tts/tokens/errors.jsonl
/content/amharic_tts/train_config_poc.json
/content/amharic_tts/manifests/train.jsonl
/content/amharic_tts/manifests/test.jsonl
/content/amharic_tts/manifests/validation.jsonl


In [25]:
!find /content/amharic_tts/tokens -maxdepth 3 -type f -print 2>/dev/null

/content/amharic_tts/tokens/train/shard-000001.tar
/content/amharic_tts/tokens/train/shard-000031.tar
/content/amharic_tts/tokens/train/shard-000026.tar
/content/amharic_tts/tokens/train/shard-000006.jsonl
/content/amharic_tts/tokens/train/shard-000018.tar
/content/amharic_tts/tokens/train/shard-000029.tar
/content/amharic_tts/tokens/train/shard-000016.jsonl
/content/amharic_tts/tokens/train/shard-000009.tar
/content/amharic_tts/tokens/train/shard-000024.jsonl
/content/amharic_tts/tokens/train/shard-000020.tar
/content/amharic_tts/tokens/train/shard-000004.jsonl
/content/amharic_tts/tokens/train/shard-000019.tar
/content/amharic_tts/tokens/train/shard-000031.jsonl
/content/amharic_tts/tokens/train/shard-000008.tar
/content/amharic_tts/tokens/train/shard-000012.tar
/content/amharic_tts/tokens/train/shard-000032.jsonl
/content/amharic_tts/tokens/train/shard-000023.tar
/content/amharic_tts/tokens/train/shard-000029.jsonl
/content/amharic_tts/tokens/train/shard-000015.jsonl
/content/amhari

In [26]:
!head -n 10 /content/amharic_tts/tokens/data.lst

/content/amharic_tts/tokens/dev/shard-000000.tar /content/amharic_tts/tokens/dev/shard-000000.jsonl 47 541.034
/content/amharic_tts/tokens/dev/shard-000001.tar /content/amharic_tts/tokens/dev/shard-000001.jsonl 47 502.314
/content/amharic_tts/tokens/dev/shard-000002.tar /content/amharic_tts/tokens/dev/shard-000002.jsonl 47 530.154
/content/amharic_tts/tokens/dev/shard-000003.tar /content/amharic_tts/tokens/dev/shard-000003.jsonl 47 502.815
/content/amharic_tts/tokens/dev/shard-000004.tar /content/amharic_tts/tokens/dev/shard-000004.jsonl 47 521.168
/content/amharic_tts/tokens/dev/shard-000005.tar /content/amharic_tts/tokens/dev/shard-000005.jsonl 47 481.435
/content/amharic_tts/tokens/dev/shard-000006.tar /content/amharic_tts/tokens/dev/shard-000006.jsonl 47 527.795
/content/amharic_tts/tokens/dev/shard-000007.tar /content/amharic_tts/tokens/dev/shard-000007.jsonl 47 532.834
/content/amharic_tts/tokens/dev/shard-000008.tar /content/amharic_tts/tokens/dev/shard-000008.jsonl 47 504.534
/

In [27]:
!wc -l /content/amharic_tts/tokens/data.lst

33 /content/amharic_tts/tokens/data.lst


In [29]:
!cat /content/amharic_tts/data_config.json

{
  "train": [
    {
      "language_id": "am",
      "manifest_path": [
        "/content/amharic_tts/tokens/train/data.lst"
      ],
      "repeat": 1
    }
  ],
  "dev": [
    {
      "language_id": "am",
      "manifest_path": [
        "/content/amharic_tts/tokens/dev/data.lst"
      ],
      "repeat": 1
    }
  ]
}

In [28]:
!wc -l /content/amharic_tts/tokens/errors.jsonl
!head -n 5 /content/amharic_tts/tokens/errors.jsonl

0 /content/amharic_tts/tokens/errors.jsonl


In [40]:
from pathlib import Path

def create_data_lst(token_dir):
    token_dir = Path(token_dir)
    tar_files = sorted(token_dir.glob("shard-*.tar"))

    lines = []

    for tar_path in tar_files:
        jsonl_path = tar_path.with_suffix(".jsonl")

        if not jsonl_path.exists():
            print(f"WARNING: missing {jsonl_path}")
            continue

        # Count examples and total duration from the JSONL
        import json

        count = 0
        duration = 0.0

        with open(jsonl_path, "r") as f:
            for line in f:
                if not line.strip():
                    continue

                item = json.loads(line)
                count += 1

                # Handle common duration field names
                duration += float(
                    item.get("duration", item.get("duration_s", 0))
                )

        lines.append(
            f"{tar_path} {jsonl_path} {count} {duration:.3f}"
        )

    output = token_dir / "data.lst"

    with open(output, "w") as f:
        f.write("\n".join(lines) + "\n")

    print(f"Created: {output}")
    print(f"Shards: {len(lines)}")


create_data_lst("/content/amharic_tts/tokens/train")

Created: /content/amharic_tts/tokens/train/data.lst
Shards: 33


In [41]:
create_data_lst("/content/amharic_tts/tokens/dev")

Created: /content/amharic_tts/tokens/dev/data.lst
Shards: 33


In [42]:
!head -n 3 /content/amharic_tts/tokens/train/data.lst

/content/amharic_tts/tokens/train/shard-000000.tar /content/amharic_tts/tokens/train/shard-000000.jsonl 240 0.000
/content/amharic_tts/tokens/train/shard-000001.tar /content/amharic_tts/tokens/train/shard-000001.jsonl 240 0.000
/content/amharic_tts/tokens/train/shard-000002.tar /content/amharic_tts/tokens/train/shard-000002.jsonl 240 0.000


In [43]:
!sed -n '1,15p' /content/amharic_tts/tokens/data.lst

/content/amharic_tts/tokens/dev/shard-000000.tar /content/amharic_tts/tokens/dev/shard-000000.jsonl 47 541.034
/content/amharic_tts/tokens/dev/shard-000001.tar /content/amharic_tts/tokens/dev/shard-000001.jsonl 47 502.314
/content/amharic_tts/tokens/dev/shard-000002.tar /content/amharic_tts/tokens/dev/shard-000002.jsonl 47 530.154
/content/amharic_tts/tokens/dev/shard-000003.tar /content/amharic_tts/tokens/dev/shard-000003.jsonl 47 502.815
/content/amharic_tts/tokens/dev/shard-000004.tar /content/amharic_tts/tokens/dev/shard-000004.jsonl 47 521.168
/content/amharic_tts/tokens/dev/shard-000005.tar /content/amharic_tts/tokens/dev/shard-000005.jsonl 47 481.435
/content/amharic_tts/tokens/dev/shard-000006.tar /content/amharic_tts/tokens/dev/shard-000006.jsonl 47 527.795
/content/amharic_tts/tokens/dev/shard-000007.tar /content/amharic_tts/tokens/dev/shard-000007.jsonl 47 532.834
/content/amharic_tts/tokens/dev/shard-000008.tar /content/amharic_tts/tokens/dev/shard-000008.jsonl 47 504.534
/

we can simply split the existing /content/amharic_tts/tokens/data.lst into the required train/dev manifests.

In [46]:
from pathlib import Path

source = Path("/content/amharic_tts/tokens/data.lst")
train_lst = Path("/content/amharic_tts/tokens/train/data.lst")
dev_lst = Path("/content/amharic_tts/tokens/dev/data.lst")

lines = source.read_text().splitlines()

train_lines = [line for line in lines if "/tokens/train/" in line]
dev_lines = [line for line in lines if "/tokens/dev/" in line]

train_lst.write_text("\n".join(train_lines) + "\n")
dev_lst.write_text("\n".join(dev_lines) + "\n")

print(f"Train shards: {len(train_lines)}")
print(f"Dev shards:   {len(dev_lines)}")
print(f"Created: {train_lst}")
print(f"Created: {dev_lst}")

Train shards: 0
Dev shards:   33
Created: /content/amharic_tts/tokens/train/data.lst
Created: /content/amharic_tts/tokens/dev/data.lst


In [49]:
!cat /content/amharic_tts/tokens/train/data.lst | head -n 5

/content/amharic_tts/tokens/train/shard-000000.tar /content/amharic_tts/tokens/train/shard-000000.jsonl 240 0.000
/content/amharic_tts/tokens/train/shard-000001.tar /content/amharic_tts/tokens/train/shard-000001.jsonl 240 0.000
/content/amharic_tts/tokens/train/shard-000002.tar /content/amharic_tts/tokens/train/shard-000002.jsonl 240 0.000
/content/amharic_tts/tokens/train/shard-000003.tar /content/amharic_tts/tokens/train/shard-000003.jsonl 240 0.000
/content/amharic_tts/tokens/train/shard-000004.tar /content/amharic_tts/tokens/train/shard-000004.jsonl 240 0.000


In [45]:
!cat /content/amharic_tts/tokens/dev/data.lst | head -n 5

/content/amharic_tts/tokens/dev/shard-000000.tar /content/amharic_tts/tokens/dev/shard-000000.jsonl 47 0.000
/content/amharic_tts/tokens/dev/shard-000001.tar /content/amharic_tts/tokens/dev/shard-000001.jsonl 47 0.000
/content/amharic_tts/tokens/dev/shard-000002.tar /content/amharic_tts/tokens/dev/shard-000002.jsonl 47 0.000
/content/amharic_tts/tokens/dev/shard-000003.tar /content/amharic_tts/tokens/dev/shard-000003.jsonl 47 0.000
/content/amharic_tts/tokens/dev/shard-000004.tar /content/amharic_tts/tokens/dev/shard-000004.jsonl 47 0.000


In [48]:
from pathlib import Path

train_dir = Path("/content/amharic_tts/tokens/train")
train_lst = train_dir / "data.lst"

# Find all train shards
tar_files = sorted(train_dir.glob("shard-*.tar"))

lines = []

for tar_path in tar_files:
    jsonl_path = tar_path.with_suffix(".jsonl")

    if not jsonl_path.exists():
        print(f"WARNING: missing {jsonl_path}")
        continue

    # Read the already-generated shard metadata.
    # We only need the number of samples and duration.
    import json

    count = 0
    duration = 0.0

    with open(jsonl_path, "r") as f:
        for line in f:
            if not line.strip():
                continue

            item = json.loads(line)
            count += 1

            # Try the fields used by the tokenizer output.
            if "duration" in item:
                duration += float(item["duration"])
            elif "duration_s" in item:
                duration += float(item["duration_s"])

    lines.append(
        f"{tar_path} {jsonl_path} {count} {duration:.3f}"
    )

train_lst.write_text("\n".join(lines) + "\n")

print(f"Created: {train_lst}")
print(f"Number of shards: {len(lines)}")

Created: /content/amharic_tts/tokens/train/data.lst
Number of shards: 33


In [50]:
!head -n 2 /content/amharic_tts/tokens/train/shard-000000.jsonl

{"id": "clip_413a70f15590", "audio_path": "/content/drive/MyDrive/amharic_tts/audio/train/clip_413a70f15590.wav", "text": "ይሁንና ሜሪጆይ ኢትዮጵያን ጨምሮ በርካታ በጎ ፈቃደኛ ወገኖች ያደረጉለት የመጠለያ፣ የቀለብና የመማሪያ ድጋፍ ተስፋው እንዲለመልም ማድረጉን ገልጿል።", "language_id": "am", "audio_duration": 10.2335, "num_tokens": 256}
{"id": "clip_e47a3698adeb", "audio_path": "/content/drive/MyDrive/amharic_tts/audio/train/clip_e47a3698adeb.wav", "text": "በትውልድ ቅብብሎሽ ጠብቆ ለማቆየት የሁሉም ሃላፊነት ሊሆን ይገባል ሲሉ አስገንዝበዋል።", "language_id": "am", "audio_duration": 4.6735, "num_tokens": 117}


In [ ]:
!accelerate launch \
  --num_processes 1 \
  -m omnivoice.cli.train \
  --train_config /content/amharic_tts/train_config_poc.json \
  --data_config /content/amharic_tts/data_config.json \
  --output_dir /content/drive/MyDrive/amharic_tts/checkpoints/poc

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files: 100% 11/11 [00:00<00:00, 1556.17it/s]
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Download complete: :           |  0.00B            
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files: 100% 11/11 [00:00<00:00, 1473.94it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 313/313 [00:00<00:00, 3583